# Variational ansatzes

This page explains how ffsim represents variational ansatz operators.

## Design

ffsim includes [several classes](#ansatz-classes) for representing variational ansatzes.
Each ansatz class stores the parameters of the ansatz as NumPy arrays, and
implements the [SupportsApplyUnitary](../explanations/protocols.ipynb) protocol so that it can be
passed to [ffsim.apply_unitary](../api/ffsim.rst#ffsim.apply_unitary) to apply the ansatz to a state vector.

To facilitate variational optimization, each ansatz class implements:

- `n_params`: a static method that returns the number of real-valued parameters
  for a given set of ansatz settings.
- `from_parameters`: a static method that constructs the ansatz from a flat NumPy array of real-valued parameters.
- `to_parameters`: an instance method that converts the ansatz back to a flat NumPy array of real-valued parameters.

These methods make it straightforward to use the ansatzes with standard numerical optimizers.

The following code sets up some objects that will be used throughout this page.

In [ ]:
import numpy as np

import ffsim

# Use 4 spatial orbitals and 2 alpha + 2 beta electrons, as an example.
norb = 4
nelec = (2, 2)
nocc = 2
nvrt = norb - nocc

rng = np.random.default_rng(1234)

# Reference state: Hartree-Fock state
reference_state = ffsim.hartree_fock_state(norb, nelec)

## Ansatz classes

### UCJ ansatzes

The unitary cluster Jastrow (UCJ) ansatz has the form

$$
  \lvert \Psi \rangle = \prod_{k = 1}^{L} \mathcal{U}_k e^{i \mathcal{J}_k} \mathcal{U}_k^\dagger \lvert \Phi_0 \rangle
$$

where $\lvert \Phi_0 \rangle$ is a reference state, each $\mathcal{U}_k$ is an
[orbital rotation](orbital-rotation.ipynb), and each $\mathcal{J}_k$ is a diagonal
Coulomb operator of the form

$$
    \mathcal{J} = \frac{1}{2}\sum_{\sigma \tau, ij} \mathbf{J}^{\sigma \tau}_{ij} n_{\sigma, i} n_{\tau, j}.
$$

The number of terms $L$ is referred to as the number of ansatz repetitions.
An optional final orbital rotation can be appended at the end to support
variational optimization of the orbital basis.

ffsim provides three UCJ classes that differ in their spin symmetry assumptions.
The diagonal Coulomb matrices and orbital rotations are stored as NumPy arrays.
The parameter vector encodes the independent entries of the diagonal Coulomb matrices
(upper-triangular entries, since the matrices are symmetric) and the entries of the
logarithms of the orbital rotations.

For more detail on the UCJ ansatz, including its connection to CCSD and the
local UCJ (LUCJ) variant, see [The local unitary cluster Jastrow (LUCJ) ansatz](lucj.ipynb).

#### UCJOpSpinBalanced

The [UCJOpSpinBalanced](../api/ffsim.rst#ffsim.UCJOpSpinBalanced) class represents the
spin-balanced UCJ operator, appropriate for closed-shell systems.
It imposes the constraints $\mathbf{J}^{\alpha\alpha} = \mathbf{J}^{\beta\beta}$
and $\mathbf{J}^{\alpha\beta} = \mathbf{J}^{\beta\alpha}$,
so each diagonal Coulomb operator is described by two symmetric matrices:
$\mathbf{J}^{\alpha\alpha}$ and $\mathbf{J}^{\alpha\beta}$.
The same orbital rotation is applied to both spin sectors.

The data stored by this class consists of:

- `diag_coulomb_mats`: An array of shape `(n_reps, 2, norb, norb)` containing the
  $\mathbf{J}^{\alpha\alpha}$ and $\mathbf{J}^{\alpha\beta}$ matrices for each repetition.
- `orbital_rotations`: An array of shape `(n_reps, norb, norb)` containing the
  orbital rotation for each repetition.
- `final_orbital_rotation`: An optional array of shape `(norb, norb)` containing
  a final orbital rotation.

In [ ]:
n_reps = 2

# Construct from a random parameter vector
n_params = ffsim.UCJOpSpinBalanced.n_params(norb, n_reps)
params = rng.standard_normal(n_params)
ucj_op = ffsim.UCJOpSpinBalanced.from_parameters(params, norb=norb, n_reps=n_reps)

print("diag_coulomb_mats shape:", ucj_op.diag_coulomb_mats.shape)
print("orbital_rotations shape:", ucj_op.orbital_rotations.shape)

# Apply the ansatz to the reference state
final_state = ffsim.apply_unitary(reference_state, ucj_op, norb=norb, nelec=nelec)

The `interaction_pairs` argument can be used to restrict the orbital interactions
in the diagonal Coulomb operators (setting the remaining entries to zero),
which yields the local UCJ (LUCJ) variant:

In [ ]:
# Restrict to nearest-neighbor (line topology) interactions
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p) for p in range(norb)]
interaction_pairs = (pairs_aa, pairs_ab)

n_params_lucj = ffsim.UCJOpSpinBalanced.n_params(
    norb, n_reps, interaction_pairs=interaction_pairs
)
params_lucj = rng.standard_normal(n_params_lucj)
lucj_op = ffsim.UCJOpSpinBalanced.from_parameters(
    params_lucj, norb=norb, n_reps=n_reps, interaction_pairs=interaction_pairs
)

final_state_lucj = ffsim.apply_unitary(reference_state, lucj_op, norb=norb, nelec=nelec)

#### UCJOpSpinUnbalanced

The [UCJOpSpinUnbalanced](../api/ffsim.rst#ffsim.UCJOpSpinUnbalanced) class represents the
spin-unbalanced UCJ operator, appropriate for open-shell systems.
It allows $\mathbf{J}^{\alpha\alpha}$ and $\mathbf{J}^{\beta\beta}$ to differ
and does not require $\mathbf{J}^{\alpha\beta}$ to be symmetric
(since $\mathbf{J}^{\alpha\beta}_{ij} = \mathbf{J}^{\beta\alpha}_{ji}$,
the $\mathbf{J}^{\beta\alpha}$ matrix need not be stored separately).
Each diagonal Coulomb operator is therefore described by three matrices:
$\mathbf{J}^{\alpha\alpha}$, $\mathbf{J}^{\alpha\beta}$, and $\mathbf{J}^{\beta\beta}$.
Separate orbital rotations are applied to the alpha and beta spin sectors.

The data stored by this class consists of:

- `diag_coulomb_mats`: An array of shape `(n_reps, 3, norb, norb)` containing the
  $\mathbf{J}^{\alpha\alpha}$, $\mathbf{J}^{\alpha\beta}$, and $\mathbf{J}^{\beta\beta}$
  matrices for each repetition.
- `orbital_rotations`: An array of shape `(n_reps, 2, norb, norb)` containing the
  alpha and beta orbital rotations for each repetition.
- `final_orbital_rotation`: An optional array of shape `(2, norb, norb)` containing
  a final orbital rotation for each spin sector.

In [ ]:
n_params = ffsim.UCJOpSpinUnbalanced.n_params(norb, n_reps)
params = rng.standard_normal(n_params)
ucj_op_unbalanced = ffsim.UCJOpSpinUnbalanced.from_parameters(
    params, norb=norb, n_reps=n_reps
)

print("diag_coulomb_mats shape:", ucj_op_unbalanced.diag_coulomb_mats.shape)
print("orbital_rotations shape:", ucj_op_unbalanced.orbital_rotations.shape)

final_state = ffsim.apply_unitary(
    reference_state, ucj_op_unbalanced, norb=norb, nelec=nelec
)

#### UCJOpSpinless

The [UCJOpSpinless](../api/ffsim.rst#ffsim.UCJOpSpinless) class represents the
spinless UCJ operator, for systems without a spin degree of freedom.
The diagonal Coulomb operator takes the form

$$
    \mathcal{J} = \frac{1}{2}\sum_{ij} \mathbf{J}_{ij} n_i n_j
$$

where $\mathbf{J}$ is a single real symmetric matrix.

The data stored by this class consists of:

- `diag_coulomb_mats`: An array of shape `(n_reps, norb, norb)` containing the
  diagonal Coulomb matrix for each repetition.
- `orbital_rotations`: An array of shape `(n_reps, norb, norb)` containing the
  orbital rotation for each repetition.
- `final_orbital_rotation`: An optional array of shape `(norb, norb)` containing
  a final orbital rotation.

In [ ]:
nelec_spinless = 2
reference_state_spinless = ffsim.hartree_fock_state(norb, nelec_spinless)

n_params = ffsim.UCJOpSpinless.n_params(norb, n_reps)
params = rng.standard_normal(n_params)
ucj_op_spinless = ffsim.UCJOpSpinless.from_parameters(params, norb=norb, n_reps=n_reps)

print("diag_coulomb_mats shape:", ucj_op_spinless.diag_coulomb_mats.shape)
print("orbital_rotations shape:", ucj_op_spinless.orbital_rotations.shape)

final_state_spinless = ffsim.apply_unitary(
    reference_state_spinless, ucj_op_spinless, norb=norb, nelec=nelec_spinless
)

### UCCSD ansatzes

The unitary coupled cluster singles and doubles (UCCSD) ansatz has the form

$$
  \lvert \Psi \rangle = e^{T - T^\dagger} \lvert \Phi_0 \rangle
$$

where

$$
  T = T_1 + T_2, \quad
  T_1 = \sum_{ia} t_{ia} a^\dagger_a a_i, \quad
  T_2 = \sum_{ijab} t_{ijab} a^\dagger_a a^\dagger_b a_j a_i.
$$

Here $i, j$ index occupied orbitals, $a, b$ index virtual orbitals, and the
$t$-amplitudes $t_{ia}$ (singles) and $t_{ijab}$ (doubles) are the variational parameters.

The UCCSD ansatz is the quantum counterpart of CCSD. Initializing a UCCSD ansatz
with $t$-amplitudes from a classical CCSD calculation provides a physically
motivated starting point for variational optimization.

An optional final orbital rotation can be appended at the end.

ffsim provides three UCCSD classes covering real/complex amplitudes and
restricted/unrestricted spin treatments.

#### UCCSDOpRestrictedReal

The [UCCSDOpRestrictedReal](../api/ffsim.rst#ffsim.UCCSDOpRestrictedReal) class represents
the restricted UCCSD operator with real-valued $t$-amplitudes.
"Restricted" means that the same spatial orbitals are used for both spin sectors,
and the same $t$-amplitudes apply to both.
This is the most common case when starting from a restricted Hartree-Fock (RHF) reference.

The data stored by this class consists of:

- `t1`: The real-valued $t_1$-amplitudes, an array of shape `(nocc, nvrt)`.
- `t2`: The real-valued $t_2$-amplitudes, an array of shape `(nocc, nocc, nvrt, nvrt)`.
- `final_orbital_rotation`: An optional array of shape `(norb, norb)` (may be complex-valued).

In [ ]:
t1 = rng.standard_normal((nocc, nvrt))
t2 = rng.standard_normal((nocc, nocc, nvrt, nvrt))
uccsd_op = ffsim.UCCSDOpRestrictedReal(t1=t1, t2=t2)

final_state = ffsim.apply_unitary(reference_state, uccsd_op, norb=norb, nelec=nelec)

The `from_parameters` method maps a flat real parameter vector to a
`UCCSDOpRestrictedReal` instance. The $t_1$-amplitudes occupy the first
`nocc * nvrt` entries, and the $t_2$-amplitudes occupy the remaining entries
(stored as the upper triangle of the outer product of occupied-virtual index pairs,
to exploit the symmetry $t_{ijab} = t_{jiab}$ imposed by the parameterization):

In [ ]:
n_params = ffsim.UCCSDOpRestrictedReal.n_params(norb, nocc)
params = rng.standard_normal(n_params)
uccsd_op = ffsim.UCCSDOpRestrictedReal.from_parameters(params, norb=norb, nocc=nocc)

# Round-trip: to_parameters recovers the parameter vector
params_recovered = uccsd_op.to_parameters()
np.testing.assert_allclose(params_recovered, params)

#### UCCSDOpRestricted

The [UCCSDOpRestricted](../api/ffsim.rst#ffsim.UCCSDOpRestricted) class is the complex-valued
counterpart of `UCCSDOpRestrictedReal`.
It allows complex $t$-amplitudes, which doubles the number of real parameters.
The real and imaginary parts of each amplitude are stored as consecutive entries
in the parameter vector.

The data stored by this class consists of:

- `t1`: The complex-valued $t_1$-amplitudes, an array of shape `(nocc, nvrt)`.
- `t2`: The complex-valued $t_2$-amplitudes, an array of shape `(nocc, nocc, nvrt, nvrt)`.
- `final_orbital_rotation`: An optional array of shape `(norb, norb)`.

In [ ]:
n_params = ffsim.UCCSDOpRestricted.n_params(norb, nocc)
params = rng.standard_normal(n_params)
uccsd_op_complex = ffsim.UCCSDOpRestricted.from_parameters(params, norb=norb, nocc=nocc)

print("t1 dtype:", uccsd_op_complex.t1.dtype)
print("t2 dtype:", uccsd_op_complex.t2.dtype)

final_state = ffsim.apply_unitary(
    reference_state, uccsd_op_complex, norb=norb, nelec=nelec
)

#### UCCSDOpUnrestrictedReal

The [UCCSDOpUnrestrictedReal](../api/ffsim.rst#ffsim.UCCSDOpUnrestrictedReal) class represents
the unrestricted UCCSD operator with real-valued $t$-amplitudes.
"Unrestricted" means that separate $t$-amplitudes are used for different spin combinations,
suitable for open-shell systems described by an unrestricted Hartree-Fock (UHF) reference.

The data stored by this class consists of:

- `t1`: A tuple `(t1_a, t1_b)` of real-valued arrays with shapes
  `(nocc_a, nvrt_a)` and `(nocc_b, nvrt_b)`.
- `t2`: A tuple `(t2_aa, t2_ab, t2_bb)` of real-valued arrays with shapes
  `(nocc_a, nocc_a, nvrt_a, nvrt_a)`, `(nocc_a, nocc_b, nvrt_a, nvrt_b)`,
  and `(nocc_b, nocc_b, nvrt_b, nvrt_b)`.
- `final_orbital_rotation`: An optional array of shape `(2, norb, norb)` containing
  a final orbital rotation for each spin sector.

In [ ]:
# Use an open-shell example: 3 alpha electrons and 1 beta electron
nelec_open = (3, 1)
nocc_a, nocc_b = nelec_open
nvrt_a = norb - nocc_a
nvrt_b = norb - nocc_b

n_params = ffsim.UCCSDOpUnrestrictedReal.n_params(norb, nelec_open)
params = rng.standard_normal(n_params)
uccsd_op_unrestricted = ffsim.UCCSDOpUnrestrictedReal.from_parameters(
    params, norb=norb, nelec=nelec_open
)

print("t1_a shape:", uccsd_op_unrestricted.t1[0].shape)
print("t1_b shape:", uccsd_op_unrestricted.t1[1].shape)
print("t2_aa shape:", uccsd_op_unrestricted.t2[0].shape)
print("t2_ab shape:", uccsd_op_unrestricted.t2[1].shape)
print("t2_bb shape:", uccsd_op_unrestricted.t2[2].shape)

reference_state_open = ffsim.hartree_fock_state(norb, nelec_open)
final_state = ffsim.apply_unitary(
    reference_state_open, uccsd_op_unrestricted, norb=norb, nelec=nelec_open
)